In [5]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse.linalg import LinearOperator, eigsh
from torch.utils.data import DataLoader
from scripts.utils import create_synthetic_dataset, FlatMLP, train_model

# ── Dataset & DataLoaders ──────────────────────────────────────────────────────
dataset = create_synthetic_dataset()
input_dim = dataset.tensors[0].shape[1]
criterion = torch.nn.CrossEntropyLoss()
full_dataloader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)

# ── Shared helper: Hessian-vector product ─────────────────────────────────────
def compute_hvp(model, dataset, v_np, device, batch_size=500):
    """Compute H·v without forming H, via autograd second-order differentials."""
    v_tensor = torch.tensor(v_np, dtype=torch.float32, device=device)
    hvp_accum = torch.zeros_like(v_tensor)
    crit = torch.nn.CrossEntropyLoss()
    for X_b, y_b in DataLoader(dataset, batch_size=batch_size, shuffle=False):
        X_b, y_b = X_b.to(device), y_b.to(device)
        model.zero_grad()
        loss = crit(model(X_b), y_b)
        grads = torch.autograd.grad(loss, model.parameters(), create_graph=True)
        flat_grad = torch.cat([g.contiguous().view(-1) for g in grads])
        grad_v = torch.dot(flat_grad, v_tensor)
        hvp_grads = torch.autograd.grad(grad_v, model.parameters())
        flat_hvp = torch.cat([g.contiguous().view(-1) for g in hvp_grads])
        hvp_accum += flat_hvp * len(X_b)
    return (hvp_accum / len(dataset)).detach().cpu().numpy()

# ── Shared helper: flatten / restore weights ──────────────────────────────────
def get_flat_weights(model):
    return torch.cat([p.data.contiguous().view(-1) for p in model.parameters()]).clone()

def set_flat_weights(model, flat_w):
    idx = 0
    for p in model.parameters():
        n = p.numel()
        p.data.copy_(flat_w[idx:idx + n].view_as(p.data))
        idx += n

# ── Shared helper: full-batch loss ────────────────────────────────────────────
def eval_loss(model, X, y, criterion):
    model.eval()
    with torch.no_grad():
        return criterion(model(X), y).item()

print("Setup complete.")

Setup complete.


In [ ]:
print("Training model to convergence (theta*)...")

trained_model = train_model(
    dataset=dataset,
    input_dim=input_dim,
    lr=0.005,
    batch_size=64,
    epochs=250
)
trained_model.eval()
device = next(trained_model.parameters()).device
X_all = dataset.tensors[0].to(device)
y_all = dataset.tensors[1].to(device)

# Baseline loss and flattened weight snapshot at theta*
with torch.no_grad():
    loss_baseline = criterion(trained_model(X_all), y_all).item()
theta_star = get_flat_weights(trained_model)

print(f"Equilibration complete!  L* = {loss_baseline:.5f}")

--- Step 4: Verification of the Quadratic Local Approximation ---


NameError: name 'evals' is not defined

In [ ]:
K = 5  # Number of top eigenvectors to compute
print(f"Computing top-{K} Hessian eigenvectors via Lanczos (eigsh)...")

M = sum(p.numel() for p in trained_model.parameters())

def _hvp_op(v_np):
    return compute_hvp(trained_model, dataset, v_np, device, batch_size=500)

H_op = LinearOperator((M, M), matvec=_hvp_op)
evals_raw, evecs_raw = eigsh(H_op, k=K, which='LA')

# eigsh returns eigenvalues in ascending order; reverse so index 0 = largest
sort_idx = np.argsort(evals_raw)[::-1]
evals = evals_raw[sort_idx].copy()
hessian_eigenvectors = evecs_raw[:, sort_idx].T.copy()  # shape (K, M)

lambda_1 = float(evals[0])
v_1      = hessian_eigenvectors[0]  # top eigenvector

print(f"Done.  λ₁ = {lambda_1:.4f}")
for i, lam in enumerate(evals):
    print(f"  λ_{i+1} = {lam:.4f}")

In [ ]:
print("--- Verification of the Quadratic Local Approximation ---")

alphas = np.linspace(-0.05, 0.05, 50)
true_losses      = []
quadratic_losses = []

v1_t = torch.tensor(v_1, dtype=torch.float32, device=device)

print("Scanning loss landscape along the steepest Hessian eigenvector...")
for alpha in alphas:
    quadratic_losses.append(loss_baseline + 0.5 * lambda_1 * alpha ** 2)
    set_flat_weights(trained_model, theta_star + alpha * v1_t)
    true_losses.append(eval_loss(trained_model, X_all, y_all, criterion))

set_flat_weights(trained_model, theta_star)  # restore theta*

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(alphas, true_losses,      'b-',  linewidth=3, label="True Empirical Loss")
ax.plot(alphas, quadratic_losses, 'r--', linewidth=3, label="Theoretical Quadratic Approximation")
ax.set_title("Local Minimum vs. Quadratic Parabola (Steepest Direction)", fontsize=14)
ax.set_xlabel(r"Perturbation Step $\alpha$ along Eigenvector $H_1$", fontsize=12)
ax.set_ylabel("Cross-Entropy Loss", fontsize=12)
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig("Quadratic_Approximation.png", dpi=300)
plt.show()

In [ ]:
print("--- Statistical Goodness-of-Fit Test for Quadratic Approximation ---")

alphas_arr      = np.array(alphas)
true_losses_arr = np.array(true_losses)

# 1. Fit a degree-2 polynomial to the empirical data
coeffs = np.polyfit(alphas_arr, true_losses_arr, deg=2)
a_fit, b_fit, c_fit = coeffs
quad_fitted = np.polyval(coeffs, alphas_arr)

# 2. R² — how much variance is explained by a quadratic
ss_res    = np.sum((true_losses_arr - quad_fitted) ** 2)
ss_tot    = np.sum((true_losses_arr - np.mean(true_losses_arr)) ** 2)
r_squared = 1.0 - ss_res / ss_tot

# 3. Compare fitted curvature against theoretical prediction (lambda_1 / 2)
a_theory = lambda_1 / 2.0
b_theory = 0.0
c_theory = loss_baseline
rel_err  = abs(a_fit - a_theory) / abs(a_theory) * 100

print(f"\nFitted Quadratic:  {a_fit:.6f}·α²  +  {b_fit:.6f}·α  +  {c_fit:.6f}")
print(f"Theoretical:       {a_theory:.6f}·α²  +  {b_theory:.6f}·α  +  {c_theory:.6f}")
print(f"\nR²                        = {r_squared:.8f}  (1.0 = perfect quadratic fit)")
print(f"Relative error on λ₁/2    = {rel_err:.4f}%")

residuals = true_losses_arr - quad_fitted

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: residual plot
axes[0].scatter(alphas_arr, residuals, color='purple', s=40, zorder=5)
axes[0].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes[0].set_title(f"Residuals from Quadratic Fit  (R² = {r_squared:.6f})", fontsize=13)
axes[0].set_xlabel(r"Perturbation $\alpha$", fontsize=12)
axes[0].set_ylabel("Residual  (True − Fitted)", fontsize=12)
axes[0].grid(True, linestyle='--', alpha=0.6)

# Right: three-way comparison
axes[1].plot(alphas_arr, true_losses_arr, 'b-', linewidth=2.5, label="True Empirical Loss")
axes[1].plot(alphas_arr, np.array(quadratic_losses), 'r--', linewidth=2.5,
             label=r"Theoretical  $L^* + \frac{\lambda_1}{2}\alpha^2$")
axes[1].plot(alphas_arr, quad_fitted, 'g:', linewidth=2.5,
             label=f"Best-Fit Quadratic  (R²={r_squared:.5f})")
axes[1].set_title("Empirical Loss vs. Quadratic Fits", fontsize=13)
axes[1].set_xlabel(r"Perturbation $\alpha$", fontsize=12)
axes[1].set_ylabel("Cross-Entropy Loss", fontsize=12)
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.savefig("Quadratic_Fit_Test.png", dpi=300)
plt.show()

In [ ]:
print("--- Quadratic Approximation: Width of Validity Region ---")

r_max      = 0.20
n_points   = 200
thresholds = [0.01, 0.05, 0.10]

alphas_wide      = np.linspace(-r_max, r_max, n_points)
true_losses_wide = []
quad_losses_wide = []

v1_t = torch.tensor(v_1, dtype=torch.float32, device=device)

with torch.no_grad():
    for alpha in alphas_wide:
        quad_losses_wide.append(loss_baseline + 0.5 * lambda_1 * alpha ** 2)
        set_flat_weights(trained_model, theta_star + alpha * v1_t)
        true_losses_wide.append(eval_loss(trained_model, X_all, y_all, criterion))

set_flat_weights(trained_model, theta_star)  # restore

true_losses_wide = np.array(true_losses_wide)
quad_losses_wide = np.array(quad_losses_wide)

abs_error = np.abs(true_losses_wide - quad_losses_wide)
with np.errstate(divide='ignore', invalid='ignore'):
    rel_error = np.where(true_losses_wide > 1e-12, abs_error / true_losses_wide, np.nan)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(alphas_wide, rel_error * 100, color='tab:blue', linewidth=2,
        label=r'Relative error  $|L_{true} - L_{quad}|\,/\,L_{true}$')

colors_thresh = ['tab:orange', 'tab:red']
for thr, col in zip(thresholds, colors_thresh):
    ax.axhline(thr * 100, color=col, linestyle='--', linewidth=1.5,
               label=f'{int(thr*100)} % threshold')
    mask = rel_error <= thr
    if mask.any():
        mid   = n_points // 2
        left  = mid
        right = mid
        while left  > 0             and mask[left  - 1]: left  -= 1
        while right < n_points - 1  and mask[right + 1]: right += 1
        ax.axvspan(alphas_wide[left], alphas_wide[right], alpha=0.08, color=col,
                   label=f'Valid region ({int(thr*100)} %): '
                         f'[{alphas_wide[left]:.3f}, {alphas_wide[right]:.3f}]')

ax.axvline(0, color='black', linestyle=':', linewidth=1)
ax.set_xlabel(r'Perturbation $\alpha$ along eigenvector $H_1$', fontsize=12)
ax.set_ylabel('Relative error  (%)', fontsize=12)
ax.set_title('Width of the Quadratic Approximation Region\n'
             r'$|L_{true}(\alpha) - L_{quad}(\alpha)|\;/\;L_{true}(\alpha)$',
             fontsize=13)
ax.set_xlim(-r_max, r_max)
ax.set_ylim(bottom=0)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("Quadratic_Validity_Region.png", dpi=300)
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from scripts.utils import FlatMLP

# ── Experiment settings ────────────────────────────────────────────────────────
learning_rates = [0.001, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03]  # η → temperature T = η/(2B)
B_fixed    = 64
num_steps  = 40000
K_power    = 40
INIT_SEED  = 0
r_max      = 0.5
n_scan     = 150
threshold  = 0.01   # 1 % relative-error criterion

print(f"Experiment: quadratic validity radius vs. temperature and λ₁")
print(f"  η values : {learning_rates}")
print(f"  B = {B_fixed},  num_steps = {num_steps},  r_max = ±{r_max}")

# ── Helpers ────────────────────────────────────────────────────────────────────
def train_fixed_steps(dataset, input_dim, lr, batch_size, num_steps, seed=INIT_SEED):
    torch.manual_seed(seed)
    m = FlatMLP(input_dim=input_dim, hidden_dim=512, num_classes=10)
    opt = torch.optim.SGD(m.parameters(), lr=lr)
    crit = torch.nn.CrossEntropyLoss()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    it = iter(loader)
    m.train()
    for _ in range(num_steps):
        try:
            X, y = next(it)
        except StopIteration:
            it = iter(loader)
            X, y = next(it)
        opt.zero_grad()
        crit(m(X), y).backward()
        opt.step()
    m.eval()
    return m

def estimate_top_eigen(model, dataset, device, num_iter=K_power):
    crit = torch.nn.CrossEntropyLoss()
    M = sum(p.numel() for p in model.parameters())
    def hvp(v_np):
        v_t = torch.tensor(v_np, dtype=torch.float32, device=device)
        accum = torch.zeros_like(v_t)
        for X_b, y_b in DataLoader(dataset, batch_size=512, shuffle=False):
            X_b, y_b = X_b.to(device), y_b.to(device)
            model.zero_grad()
            loss = crit(model(X_b), y_b)
            grads = torch.autograd.grad(loss, model.parameters(), create_graph=True)
            gv = torch.dot(torch.cat([g.contiguous().view(-1) for g in grads]), v_t)
            hg = torch.autograd.grad(gv, model.parameters())
            accum += torch.cat([g.contiguous().view(-1) for g in hg]) * len(X_b)
        return (accum / len(dataset)).detach().cpu().numpy()
    rng_pi = np.random.default_rng(1)
    v = rng_pi.standard_normal(M)
    v /= np.linalg.norm(v)
    lam = 0.0
    for _ in range(num_iter):
        v_new = hvp(v)
        lam = float(np.linalg.norm(v_new))
        v = v_new / lam
    return lam, v

def find_radius(alphas, rel_errors, threshold):
    crossings = np.where(rel_errors > threshold)[0]
    if len(crossings) == 0:
        return float(alphas[-1]), True
    i = crossings[0]
    if i == 0:
        return 0.0, False
    alpha_lo, alpha_hi = float(alphas[i - 1]), float(alphas[i])
    err_lo,   err_hi   = float(rel_errors[i - 1]), float(rel_errors[i])
    frac = (threshold - err_lo) / (err_hi - err_lo)
    return alpha_lo + frac * (alpha_hi - alpha_lo), False

# ── Main sweep ────────────────────────────────────────────────────────────────
results   = []
crit_eval = torch.nn.CrossEntropyLoss()
alphas_right = np.linspace(0, r_max, n_scan + 1)[1:]
alphas_left  = np.linspace(0, r_max, n_scan + 1)[1:]

for eta in learning_rates:
    T = eta / (2 * B_fixed)
    print(f"\n── η = {eta:.4f}  (T = {T:.6f}) ──")

    m_i    = train_fixed_steps(dataset, input_dim, lr=eta, batch_size=B_fixed, num_steps=num_steps)
    dev    = next(m_i.parameters()).device
    X_i    = dataset.tensors[0].to(dev)
    y_i    = dataset.tensors[1].to(dev)

    with torch.no_grad():
        L_star = crit_eval(m_i(X_i), y_i).item()
    print(f"  L* = {L_star:.5f}")

    lam1, v1_np = estimate_top_eigen(m_i, dataset, dev)
    print(f"  λ₁ = {lam1:.4f}")

    orig_w = get_flat_weights(m_i)
    v1_t   = torch.tensor(v1_np, dtype=torch.float32, device=dev)

    def scan_direction(alphas_pos, sign):
        errs = []
        with torch.no_grad():
            for alpha in alphas_pos:
                set_flat_weights(m_i, orig_w + sign * alpha * v1_t)
                L_true = crit_eval(m_i(X_i), y_i).item()
                L_quad = L_star + 0.5 * lam1 * alpha ** 2
                errs.append(abs(L_quad - L_true) / max(L_true, 1e-12))
        set_flat_weights(m_i, orig_w)
        return np.array(errs)

    rel_errors_right = scan_direction(alphas_right, +1)
    rel_errors_left  = scan_direction(alphas_left,  -1)

    r_right, capped_right = find_radius(alphas_right, rel_errors_right, threshold)
    r_left,  capped_left  = find_radius(alphas_left,  rel_errors_left,  threshold)
    print(f"  r_right ({int(threshold*100)}%) = {r_right:.5f}{'  [capped]' if capped_right else ''}")
    print(f"  r_left  ({int(threshold*100)}%) = {r_left:.5f}{'  [capped]' if capped_left else ''}")

    results.append(dict(eta=eta, T=T, lam1=lam1,
                        r_right=r_right, capped_right=capped_right,
                        r_left=r_left,  capped_left=capped_left))

# ── Plots ─────────────────────────────────────────────────────────────────────
temps     = [r['T']            for r in results]
lam1s     = [r['lam1']         for r in results]
r_rights  = [r['r_right']      for r in results]
r_lefts   = [r['r_left']       for r in results]
cap_right = [r['capped_right'] for r in results]
cap_left  = [r['capped_left']  for r in results]

def make_guide(lam1s, r_vals, cap_flags):
    valid = [(l, rv) for l, rv, c in zip(lam1s, r_vals, cap_flags) if not c]
    if not valid:
        return None, None
    scale = np.nanmean([rv * np.sqrt(l) for l, rv in valid])
    lam_range = np.linspace(min(lam1s) * 0.9, max(lam1s) * 1.1, 100)
    return lam_range, scale / np.sqrt(lam_range)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

for row, (r_vals, cap_flags, side_label) in enumerate([
        (r_rights, cap_right, 'Right  (+v₁)'),
        (r_lefts,  cap_left,  'Left   (−v₁)'),
]):
    colors = ['tab:red' if c else 'tab:blue' for c in cap_flags]

    ax0 = axes[row, 0]
    ax0.plot(temps, r_vals, color='tab:blue', linewidth=1.5, zorder=3)
    ax0.scatter(temps, r_vals, c=colors, s=80, zorder=5)
    ax0.set_xlabel(r'Temperature  $T = \eta\,/\,(2B)$', fontsize=12)
    ax0.set_ylabel(r'Validity radius  $r_{1\%}$', fontsize=12)
    ax0.set_title(f'[{side_label}]  Validity radius vs. Temperature', fontsize=12)
    ax0.set_xscale('log')
    ax0.grid(True, linestyle='--', alpha=0.5)
    if any(cap_flags):
        ax0.annotate('● capped at $r_{max}$', xy=(0.03, 0.05),
                     xycoords='axes fraction', color='tab:red', fontsize=9)

    ax1 = axes[row, 1]
    ax1.scatter(lam1s, r_vals, c=colors, s=80, zorder=5)
    lam_range, guide = make_guide(lam1s, r_vals, cap_flags)
    if guide is not None:
        ax1.plot(lam_range, guide, 'k--', linewidth=1.5,
                 label=r'$r \propto \lambda_1^{-1/2}$  (guide)')
    ax1.set_xlabel(r'Largest Hessian eigenvalue  $\lambda_1$', fontsize=12)
    ax1.set_ylabel(r'Validity radius  $r_{1\%}$', fontsize=12)
    ax1.set_title(f'[{side_label}]  Validity radius vs. $\\lambda_1$', fontsize=12)
    ax1.legend(fontsize=10)
    ax1.grid(True, linestyle='--', alpha=0.5)

fig.suptitle(f'Quadratic Validity Region — Right vs. Left Side of Minimum\n'
             f'(1% relative-error threshold, B={B_fixed}, {num_steps} steps)',
             fontsize=13)
plt.tight_layout()
plt.savefig("Validity_Radius_Right_Left.png", dpi=300)
plt.show()

print(f"\n{'η':>8}  {'T':>10}  {'λ₁':>8}  {'r_right':>9}  {'r_left':>9}")
for r in results:
    print(f"{r['eta']:>8.4f}  {r['T']:>10.6f}  {r['lam1']:>8.3f}"
          f"  {r['r_right']:>9.5f}{'*' if r['capped_right'] else ' '}"
          f"  {r['r_left']:>9.5f}{'*' if r['capped_left'] else ' '}")
print("  * = capped at r_max (true radius is larger)")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from scripts.utils import FlatMLP

# ── Experiment settings ────────────────────────────────────────────────────────
eta_fixed   = 0.005
batch_sizes = [16, 32, 64, 128, 256]
num_steps   = 40000
INIT_SEED   = 0
r_max       = 0.30
n_scan      = 150
threshold   = 0.05
MAX_ITER    = 300
EIGEN_TOL   = 1e-5

print(f"Experiment: quadratic validity radius vs. batch size and λ₁")
print(f"  η = {eta_fixed}  (fixed)")
print(f"  B values : {batch_sizes},  num_steps = {num_steps},  r_max = ±{r_max}")
print(f"  Power iteration: max_iter={MAX_ITER}, tol={EIGEN_TOL}")

# ── Helpers ────────────────────────────────────────────────────────────────────
def train_fixed_steps_B(dataset, input_dim, lr, batch_size, num_steps, seed=INIT_SEED):
    torch.manual_seed(seed)
    m = FlatMLP(input_dim=input_dim, hidden_dim=512, num_classes=10)
    opt = torch.optim.SGD(m.parameters(), lr=lr)
    crit = torch.nn.CrossEntropyLoss()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    it = iter(loader)
    m.train()
    for _ in range(num_steps):
        try:
            X, y = next(it)
        except StopIteration:
            it = iter(loader)
            X, y = next(it)
        opt.zero_grad()
        crit(m(X), y).backward()
        opt.step()
    m.eval()
    return m

def estimate_top_eigen_B(model, dataset, device, max_iter=MAX_ITER, tol=EIGEN_TOL):
    crit = torch.nn.CrossEntropyLoss()
    M = sum(p.numel() for p in model.parameters())
    def hvp(v_np):
        v_t = torch.tensor(v_np, dtype=torch.float32, device=device)
        accum = torch.zeros_like(v_t)
        for X_b, y_b in DataLoader(dataset, batch_size=512, shuffle=False):
            X_b, y_b = X_b.to(device), y_b.to(device)
            model.zero_grad()
            loss = crit(model(X_b), y_b)
            grads = torch.autograd.grad(loss, model.parameters(), create_graph=True)
            gv = torch.dot(torch.cat([g.contiguous().view(-1) for g in grads]), v_t)
            hg = torch.autograd.grad(gv, model.parameters())
            accum += torch.cat([g.contiguous().view(-1) for g in hg]) * len(X_b)
        return (accum / len(dataset)).detach().cpu().numpy()
    rng_pi = np.random.default_rng(1)
    v = rng_pi.standard_normal(M)
    v /= np.linalg.norm(v)
    lam_prev = 0.0
    n_iters  = 0
    for k in range(max_iter):
        v_new = hvp(v)
        lam   = float(np.linalg.norm(v_new))
        v     = v_new / lam
        n_iters = k + 1
        if lam_prev > 0 and abs(lam - lam_prev) / lam_prev < tol:
            break
        lam_prev = lam
    return lam, v, n_iters

def find_radius_B(alphas, rel_errors, threshold):
    crossings = np.where(rel_errors > threshold)[0]
    if len(crossings) == 0:
        return float(alphas[-1]), True
    i = crossings[0]
    if i == 0:
        return 0.0, False
    alpha_lo, alpha_hi = float(alphas[i - 1]), float(alphas[i])
    err_lo,   err_hi   = float(rel_errors[i - 1]), float(rel_errors[i])
    frac = (threshold - err_lo) / (err_hi - err_lo)
    return alpha_lo + frac * (alpha_hi - alpha_lo), False

# ── Main sweep ────────────────────────────────────────────────────────────────
results_B  = []
crit_eval  = torch.nn.CrossEntropyLoss()
alphas_right = np.linspace(0, r_max, n_scan + 1)[1:]
alphas_left  = np.linspace(0, r_max, n_scan + 1)[1:]

for B in batch_sizes:
    T = eta_fixed / (2 * B)
    print(f"\n── B = {B},  η = {eta_fixed:.4f}  (T = {T:.6f}) ──")

    m_i  = train_fixed_steps_B(dataset, input_dim, lr=eta_fixed, batch_size=B, num_steps=num_steps)
    dev  = next(m_i.parameters()).device
    X_i  = dataset.tensors[0].to(dev)
    y_i  = dataset.tensors[1].to(dev)

    with torch.no_grad():
        L_star = crit_eval(m_i(X_i), y_i).item()
    print(f"  L* = {L_star:.5f}")

    lam1, v1_np, n_iters = estimate_top_eigen_B(m_i, dataset, dev)
    converged = n_iters < MAX_ITER
    print(f"  λ₁ = {lam1:.4f}  ({n_iters} iters, {'converged' if converged else 'MAX ITER reached'})")

    orig_w = get_flat_weights(m_i)
    v1_t   = torch.tensor(v1_np, dtype=torch.float32, device=dev)

    def scan_direction_B(alphas_pos, sign):
        errs = []
        with torch.no_grad():
            for alpha in alphas_pos:
                set_flat_weights(m_i, orig_w + sign * alpha * v1_t)
                L_true = crit_eval(m_i(X_i), y_i).item()
                L_quad = L_star + 0.5 * lam1 * alpha ** 2
                errs.append(abs(L_quad - L_true) / max(L_true, 1e-12))
        set_flat_weights(m_i, orig_w)
        return np.array(errs)

    rel_errors_right = scan_direction_B(alphas_right, +1)
    rel_errors_left  = scan_direction_B(alphas_left,  -1)

    r_right, capped_right = find_radius_B(alphas_right, rel_errors_right, threshold)
    r_left,  capped_left  = find_radius_B(alphas_left,  rel_errors_left,  threshold)
    print(f"  r_right ({int(threshold*100)}%) = {r_right:.5f}{'  [capped]' if capped_right else ''}")
    print(f"  r_left  ({int(threshold*100)}%) = {r_left:.5f}{'  [capped]' if capped_left else ''}")

    results_B.append(dict(B=B, T=T, lam1=lam1, n_iters=n_iters, converged=converged,
                          r_right=r_right, capped_right=capped_right,
                          r_left=r_left,   capped_left=capped_left))

# ── Plots ─────────────────────────────────────────────────────────────────────
Bs        = [r['B']            for r in results_B]
lam1s_B   = [r['lam1']         for r in results_B]
r_rights  = [r['r_right']      for r in results_B]
r_lefts   = [r['r_left']       for r in results_B]
cap_right = [r['capped_right'] for r in results_B]
cap_left  = [r['capped_left']  for r in results_B]

def make_guide_B(lam1s, r_vals, cap_flags):
    valid = [(l, rv) for l, rv, c in zip(lam1s, r_vals, cap_flags) if not c]
    if not valid:
        return None, None
    scale = np.nanmean([rv * np.sqrt(l) for l, rv in valid])
    lam_range = np.linspace(min(lam1s) * 0.9, max(lam1s) * 1.1, 100)
    return lam_range, scale / np.sqrt(lam_range)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

for row, (r_vals, cap_flags, side_label) in enumerate([
        (r_rights, cap_right, 'Right  (+v₁)'),
        (r_lefts,  cap_left,  'Left   (−v₁)'),
]):
    colors = ['tab:red' if c else 'tab:blue' for c in cap_flags]

    ax0 = axes[row, 0]
    ax0.plot(Bs, r_vals, color='tab:blue', linewidth=1.5, zorder=3)
    ax0.scatter(Bs, r_vals, c=colors, s=80, zorder=5)
    ax0.set_xlabel('Batch size  $B$', fontsize=12)
    ax0.set_ylabel(r'Validity radius  $r_{5\%}$', fontsize=12)
    ax0.set_title(f'[{side_label}]  Validity radius vs. Batch size', fontsize=12)
    ax0.set_xscale('log')
    ax0.set_xticks(Bs)
    ax0.set_xticklabels(Bs)
    ax0.grid(True, linestyle='--', alpha=0.5)
    if any(cap_flags):
        ax0.annotate('● capped at $r_{max}$', xy=(0.03, 0.05),
                     xycoords='axes fraction', color='tab:red', fontsize=9)

    ax1 = axes[row, 1]
    ax1.scatter(lam1s_B, r_vals, c=colors, s=80, zorder=5)
    lam_range, guide = make_guide_B(lam1s_B, r_vals, cap_flags)
    if guide is not None:
        ax1.plot(lam_range, guide, 'k--', linewidth=1.5,
                 label=r'$r \propto \lambda_1^{-1/2}$  (guide)')
    ax1.set_xlabel(r'Largest Hessian eigenvalue  $\lambda_1$', fontsize=12)
    ax1.set_ylabel(r'Validity radius  $r_{5\%}$', fontsize=12)
    ax1.set_title(f'[{side_label}]  Validity radius vs. $\\lambda_1$', fontsize=12)
    ax1.legend(fontsize=10)
    ax1.grid(True, linestyle='--', alpha=0.5)

fig.suptitle(f'Quadratic Validity Region — Right vs. Left Side of Minimum\n'
             f'(5% relative-error threshold, η={eta_fixed}, {num_steps} steps)',
             fontsize=13)
plt.tight_layout()
plt.savefig("Validity_Radius_Bsweep_Right_Left.png", dpi=300)
plt.show()

print(f"\n{'B':>6}  {'T':>10}  {'λ₁':>8}  {'iters':>6}  {'r_right':>9}  {'r_left':>9}")
for r in results_B:
    print(f"{r['B']:>6}  {r['T']:>10.6f}  {r['lam1']:>8.3f}  {r['n_iters']:>6d}"
          f"  {r['r_right']:>9.5f}{'*' if r['capped_right'] else ' '}"
          f"  {r['r_left']:>9.5f}{'*' if r['capped_left'] else ' '}")
print("  * = capped at r_max (true radius is larger)")